In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

ratings_url = "https://raw.githubusercontent.com/lher96/DATA612/refs/heads/main/ratings.csv"

ratings = pd.read_csv(ratings_url)

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Abstract

This project builds on a previous collaborative filtering recommender system built in our last project using MovieLens rating data. Originally I compared User User and Item Item Collaborative Filtering using both raw cosine similarity and mean centered cosine similarity. For this assignment I decided to build on my best performing model from that recommender system by adding Singular Value Decomposition as a matrix factorization method. SVD transforms the original user item ratings matrix into a smaller set of latent features that represent hidden patterns in user preferences and movie characteristics. Traditional SVD requires a complete matrix and so I handled these missing ratings problem by mean based imputation and mean centering values before factorization. That reconstructed ratings matrix was then used to predict ratings in the test set and the model performance was evaluated using RMSE, MAE, and Precision@K. RMSE and MAE measure for rating prediction error and Precision@K directly evaluates whether the highest ranked recommendations are actually relevant to users. This project compares traditional neighborhood based collaborative filtering with matrix factorization and shows how SVD can be used to create a lower dimensional recommender model. 


Item Item Collaborative Filtering was our previous best model using the raw cosine as opposed to using mean but this will be approached differently since svd can not have the blank values
Similarity: Raw Cosine
k = 20
Test RMSE = 0.859551
Test MAE = 0.655545

That model is used as a benchmark and I will compare it against a new SVD matrix factorization model. I evaluate both models using RMSE, MAE, and Precision@K.

In [5]:
#split training and test data 80/20

train, test = train_test_split(
    ratings,
    test_size=0.2,
    random_state=50
)

print(train.shape)
print(test.shape)

#Create user-item matrix from training data only

user_item_matrix = train.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_item_matrix.head()

(80668, 4)
(20168, 4)


movieId,1,2,3,4,5,6,7,8,9,10,...,191005,193565,193567,193571,193573,193579,193581,193583,193585,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
#Simple statistics about the dataset

print("Number of ratings:", len(ratings))
print("Number of users:", ratings["userId"].nunique())
print("Number of movies:", ratings["movieId"].nunique())
print("Missing values:")
print(ratings.isna().sum())

Number of ratings: 100836
Number of users: 610
Number of movies: 9724
Missing values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [26]:
#Baseline values from the training data, used global mean to confirm split
#We do indeed have the same mean 
global_mean = train["rating"].mean()

user_means = train.groupby("userId")["rating"].mean()
item_means = train.groupby("movieId")["rating"].mean()

print("Global mean rating:", global_mean)

Global mean rating: 3.5015681558982497


In [27]:
#Item item raw cosine similarity matrix
matrix_filled = user_item_matrix.fillna(0)

item_similarity = cosine_similarity(matrix_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

item_similarity_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,191005,193565,193567,193571,193573,193579,193581,193583,193585,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.350310,0.240483,0.039560,0.269680,0.253116,0.256238,0.043696,0.165360,0.343365,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.350310,1.000000,0.239625,0.063982,0.214368,0.179867,0.195468,0.112792,0.038846,0.285142,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.240483,0.239625,1.000000,0.096470,0.356790,0.251338,0.413784,0.244193,0.302828,0.195655,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.039560,0.063982,0.096470,1.000000,0.199971,0.102293,0.300234,0.263371,0.000000,0.069522,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.269680,0.214368,0.356790,0.199971,1.000000,0.276970,0.485840,0.136807,0.357785,0.170793,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
#Function to run item-item prediction

def predict_item_item(user_id, movie_id, similarity_df, k=20):
    
    #If user is not in the training matrix, use the movie's average rating
    #If the movie average is not available, use the global average
    if user_id not in user_item_matrix.index:
        return item_means.get(movie_id, global_mean)
    
    #If movie is not in the training matrix, use the user's average rating
    #If the user average is not available, use the global average
    if movie_id not in user_item_matrix.columns:
        return user_means.get(user_id, global_mean)
    
    #Get all movies rated by the user
    user_ratings = user_item_matrix.loc[user_id].dropna()
    
    #Remove the target movie if the user already rated it
    user_ratings = user_ratings.drop(index=movie_id, errors="ignore")
    
    #If the user has no other ratings then use the user's average rating
    if user_ratings.empty:
        return user_means.get(user_id, global_mean)
    
    #Find similarities between the target movie and the movies rated by this user
    similarities = similarity_df.loc[movie_id, user_ratings.index]
    
    #Keep the positive similarities
    similarities = similarities[similarities > 0]
    
    #If no positively similar movies use the user's average rating
    if similarities.empty:
        return user_means.get(user_id, global_mean)
    
    #Select top k most similar movies
    top_k_sims = similarities.sort_values(ascending=False).head(k)
    
    #Get user's ratings for those top k similar movies
    top_k_ratings = user_ratings.loc[top_k_sims.index]
    
    #Weighted prediction based on similar movie ratings
    pred = np.dot(top_k_sims, top_k_ratings) / top_k_sims.sum()
    
    #Return the predicted rating
    return pred
    

In [29]:
#Create a matrix for SVD using training data only

svd_matrix = train.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

#Since SVD requires no missing values we have to either impute or remove datapoints
#I am choosing to impute and compare the methods
svd_global_filled = svd_matrix.fillna(global_mean)

svd_global_user_means = svd_global_filled.mean(axis=1)

svd_global_centered = svd_global_filled.sub(
    svd_global_user_means,
    axis=0
)

svd_global_centered.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,191005,193565,193567,193571,193573,193579,193581,193583,193585,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.481059,-0.017373,0.481059,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373,...,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373,-0.017373
2,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,...,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441,-0.001441
3,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,...,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231,0.003231
4,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,...,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193,-0.001193
5,0.497993,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,...,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439,-0.000439


## Imputation and mean centering

In doing this assignment I thought about how it built upon our last one. When I previously considered rating imputation, I thought about potential methods and the consideration of using mean centering as to help in user understanding. In reading and learning more about svd, it seemed beneficial to do this and help remove some observed user bias. I would strongly argue this method isn't best for all recommenders. I can think of examples such as time series where this would only make sense in a moving average model like ARIMA or in using an ad system where it's more rooted in user tendencies. In this case I believe for the svd to be most successful it was better done this way. An interesting test could also be using KNN and MICE imputation methods against mean imputation and also testing them centered around their own mean. These are all possibilities of comparison but I chose this method as it has performed very well in most of my previous projects.

In [30]:
#mean imputation user vs global

svd_user_filled = svd_matrix.copy()

svd_user_filled = svd_user_filled.apply(lambda row: row.fillna(row.mean()),axis=1)

svd_user_filled = svd_user_filled.fillna(global_mean)

svd_user_user_means = svd_user_filled.mean(axis=1)

svd_user_centered = svd_user_filled.sub(
    svd_user_user_means,
    axis=0
)

svd_user_centered.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,191005,193565,193567,193571,193573,193579,193581,193583,193585,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-3.743017e-01,7.256418e-13,-3.743017e-01,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,...,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13,7.256418e-13
2,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,...,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14,-4.440892e-14
3,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,...,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13,1.043610e-13
4,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,...,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13,-2.660094e-13
5,3.823529e-01,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,...,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13,-7.998047e-13


In [31]:
#apply SVD to both global and user mean imputation

U_global, sigma_global, Vt_global = np.linalg.svd(
    svd_global_centered,
    full_matrices=False
)

U_user, sigma_user, Vt_user = np.linalg.svd(
    svd_user_centered,
    full_matrices=False
)

print("Global mean SVD shapes:")
print(U_global.shape, sigma_global.shape, Vt_global.shape)

print("\nUser mean SVD shapes:")
print(U_user.shape, sigma_user.shape, Vt_user.shape)

Global mean SVD shapes:
(610, 610) (610,) (610, 8992)

User mean SVD shapes:
(610, 610) (610,) (610, 8992)


In [32]:
#svd prediction matrix function

def build_svd_prediction_matrix(k, imputation_method):
    
    #Choose SVD decomposition to use based on the imputation method.
    #Each method has its own U, sigma, Vt, and user mean values.
    
    if imputation_method == "global_mean":
        #Use the SVD components created from the global imputed matrix
        U = U_global
        sigma = sigma_global
        Vt = Vt_global
        
        #Keep user means because of centering
        user_mean_values = svd_global_user_means.values
    
    elif imputation_method == "user_mean":
        U = U_user
        sigma = sigma_user
        Vt = Vt_user
        
        #User means are needed to convert centered predictions back into normal rating predictions.
        user_mean_values = svd_user_user_means.values
    
    else:
        raise ValueError("imputation_method must be 'global_mean' or 'user_mean'")
    

    U_k = U[:, :k]
    
    sigma_k = np.diag(sigma[:k])
    
    Vt_k = Vt[:k, :]
    
    
    #Reconstruct the centered rating matrix using only the top k factors to give an approximation of the original centered matrix.
    reconstructed_centered = np.dot(
        np.dot(U_k, sigma_k),
        Vt_k
    )
    
    
    #Add each user's mean rating back to their row
    reconstructed = reconstructed_centered + user_mean_values.reshape(-1, 1)
    
    
    #Convert the reconstructed NumPy array back into a pandas DataFrame
    pred_matrix = pd.DataFrame(
        reconstructed,
        index=svd_matrix.index,
        columns=svd_matrix.columns
    )
    
    return pred_matrix

In [33]:
#function to evaluate model

def evaluate_model(data, model_type, similarity_df=None, pred_matrix=None, k=20):
    
    predictions = []
    
    for row in data.itertuples():
        user_id = row.userId
        movie_id = row.movieId
        
        if model_type == "item_item":
            pred = predict_item_item(
                user_id=user_id,
                movie_id=movie_id,
                similarity_df=similarity_df,
                k=k
            )
        
        elif model_type == "svd":
            pred = predict_svd(
                user_id=user_id,
                movie_id=movie_id,
                pred_matrix=pred_matrix
            )
        
        else:
            raise ValueError("model_type must be 'item_item' or 'svd'")
        
        predictions.append(pred)
    
    actual = data["rating"].values
    
    rmse = root_mean_squared_error(actual, predictions)
    mae = mean_absolute_error(actual, predictions)
    
    return rmse, mae, predictions

In [34]:
#function for precision at k
def precision_at_k(data, predictions, k=5, threshold=4):
    
    temp = data.copy()
    temp["prediction"] = predictions
    
    precision_scores = []
    
    for user_id, group in temp.groupby("userId"):
        
        top_k = group.sort_values(
            "prediction",
            ascending=False
        ).head(k)
        
        if len(top_k) == 0:
            continue
        
        relevant_count = (top_k["rating"] >= threshold).sum()
        
        precision = relevant_count / len(top_k)
        
        precision_scores.append(precision)
    
    return np.mean(precision_scores)

In [35]:
#running our benchmark model and current svd 

results = []

# Benchmark model from previous assignment:
# Item-Item Collaborative Filtering with Raw Cosine Similarity, k = 20

train_rmse, train_mae, train_preds = evaluate_model(
    train,
    model_type="item_item",
    similarity_df=item_similarity_df,
    k=20
)

test_rmse, test_mae, test_preds = evaluate_model(
    test,
    model_type="item_item",
    similarity_df=item_similarity_df,
    k=20
)

test_precision = precision_at_k(
    test,
    test_preds,
    k=5,
    threshold=4
)

results.append({
    "model": "Item-Item CF",
    "method": "Raw Cosine",
    "imputation": "None",
    "k": 20,
    "train_RMSE": train_rmse,
    "train_MAE": train_mae,
    "test_RMSE": test_rmse,
    "test_MAE": test_mae,
    "Precision@5": test_precision
})


# SVD models with global mean and user mean imputation

svd_k_values = [5, 10, 20, 50, 100]

max_k = min(svd_matrix.shape)

for imputation_method in ["global_mean", "user_mean"]:
    
    for k in svd_k_values:
        
        if k > max_k:
            continue
        
        pred_matrix = build_svd_prediction_matrix(
            k=k,
            imputation_method=imputation_method
        )
        
        train_rmse, train_mae, train_preds = evaluate_model(
            train,
            model_type="svd",
            pred_matrix=pred_matrix
        )
        
        test_rmse, test_mae, test_preds = evaluate_model(
            test,
            model_type="svd",
            pred_matrix=pred_matrix
        )
        
        test_precision = precision_at_k(
            test,
            test_preds,
            k=5,
            threshold=4
        )
        
        results.append({
            "model": "SVD Matrix Factorization",
            "method": "Latent Factors",
            "imputation": imputation_method,
            "k": k,
            "train_RMSE": train_rmse,
            "train_MAE": train_mae,
            "test_RMSE": test_rmse,
            "test_MAE": test_mae,
            "Precision@5": test_precision
        })

results_df = pd.DataFrame(results)

results_df

,model,method,imputation,k,train_RMSE,train_MAE,test_RMSE,test_MAE,Precision@5
0,Item-Item CF,Raw Cosine,None,20,0.856455,0.650057,0.865734,0.660124,0.689984
1,SVD Matrix Factorization,Latent Factors,global_mean,5,0.934657,0.722898,0.989580,0.779948,0.672578
2,SVD Matrix Factorization,Latent Factors,global_mean,10,0.885064,0.669314,0.985018,0.775170,0.681445
3,SVD Matrix Factorization,Latent Factors,global_mean,20,0.818324,0.599510,0.985899,0.774533,0.680131
4,SVD Matrix Factorization,Latent Factors,global_mean,50,0.679865,0.455060,0.998351,0.784673,0.671921
5,SVD Matrix Factorization,Latent Factors,global_mean,100,0.526805,0.314076,1.012474,0.798818,0.648933
6,SVD Matrix Factorization,Latent Factors,user_mean,5,0.852055,0.638365,0.915623,0.706680,0.681117
7,SVD Matrix Factorization,Latent Factors,user_mean,10,0.810386,0.595052,0.912049,0.703702,0.686043
8,SVD Matrix Factorization,Latent Factors,user_mean,20,0.752134,0.530918,0.915568,0.705811,0.690640
9,SVD Matrix Factorization,Latent Factors,user_mean,50,0.623494,0.400553,0.925420,0.714739,0.671921


## Results

First comparing this against my previous best model we were actually worse by a small margin. The Item Item model had a test RMSE of 0.8657 and a test MAE of 0.6601, which is very close to the previous result of     0.8596 RMSE and 0.6555 MAE. The small difference could be due to minor implementation differences in the prediction function or fallback rules. There are several other comparisons to be made here includig the imputation method, k value, and what metric to determine the best model. First the imputation method showed a clear trend as the best global_mean SVD test RMSE = 0.985018 at k = 10 and best user_mean SVD test RMSE = 0.912049 at k = 10. We also see this in precision@K where the user means have a higher value for almost every k and yet still fall slightly short of our orignal model. We also saw that as k value went higher, the train RMSE and MAE decreased but this trend failed to follow in our test dataset. This is likely an indication of overfitting on the training data with the SVD. Increasing the number of latent factors improved train dataset performance but eventually hurt test performance. I find this interesting and as much as it was less successful, I believe with some tweaks this model can be improved. Finally, in our best model we do see that based on test RMSE and MAE I would say that our previous model still wins. We were close but if we were only using  precision@K, SVD with user_mean imputation, k = 20 and precision@5 = 0.690640 would be our best model. This means that SVD did not actually predict exact ratings as accurately, but it was slightly better at placing relevant movies in the top five recommendations. Ultimately I would still say the benchmark Item Item CF model was the best model for rating prediction while SVD with user mean imputation was competitive for recommendation quality considering precision@K. I think there is a valid argument for using this as a strong criteria for why you may disgregard a small enough RMSE/MAE difference.